# NBA Achilles Survival — Exploratory Data Analysis

This notebook explores:
1. Ground-truth Achilles IL events from ProSportsTransactions
2. Player biometrics and demographics
3. Workload (ACWR) distributions around rupture events
4. Play-style embedding structure
5. Kaplan-Meier survival curves stratified by age / position / ACWR

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROCESSED = Path('../data/processed')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Ground-truth Achilles events

In [ ]:
achilles = pd.read_csv(PROCESSED / 'achilles_ground_truth.csv', parse_dates=['date'])
print(f'Total records: {len(achilles)}')
achilles.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Events over time
achilles.groupby(achilles['date'].dt.year).size().plot(
    ax=axes[0], title='Achilles IL placements per year'
)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

# Severity breakdown
achilles['severity'].value_counts().plot.bar(ax=axes[1], title='Severity breakdown')
axes[1].set_xlabel('Severity')
plt.tight_layout()

## 2. Player biometrics

In [ ]:
profiles_path = PROCESSED / 'bbref_player_profiles.csv'
if profiles_path.exists():
    profiles = pd.read_csv(profiles_path)
    print(profiles.shape)
    profiles[['height_inches', 'weight_lbs', 'draft_year', 'draft_pick']].describe()
else:
    print('Run scrape_bball_reference.py first')

## 3. ACWR distribution around rupture events

In [ ]:
acwr_path = PROCESSED / 'acwr_features.csv'
if acwr_path.exists():
    acwr = pd.read_csv(acwr_path, parse_dates=['game_date'])

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, scale in zip(axes, ['short', 'medium', 'long']):
        col = f'acwr_{scale}'
        if col in acwr.columns:
            acwr[col].dropna().hist(bins=50, ax=ax)
            ax.axvline(1.5, color='red', linestyle='--', label='spike threshold')
            ax.set_title(f'ACWR {scale}')
            ax.legend()
    plt.tight_layout()
else:
    print('Run features/acwr.py first')

## 4. Kaplan-Meier survival curves

In [ ]:
fm_path = PROCESSED / 'feature_matrix.csv'
if fm_path.exists():
    from lifelines import KaplanMeierFitter

    fm = pd.read_csv(fm_path)
    # One row per player: first event or end of follow-up
    player_summary = (
        fm.sort_values('observation_date')
          .groupby('player_id')
          .last()
          .reset_index()
    )

    kmf = KaplanMeierFitter()
    kmf.fit(
        player_summary['time_to_event_days'] / 365.25,
        event_observed=player_summary['event'] == 1,
        label='All players'
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    kmf.plot_survival_function(ax=ax, ci_show=True)
    ax.set_xlabel('Years in NBA')
    ax.set_ylabel('Achilles-rupture-free survival')
    ax.set_title('Kaplan-Meier: Achilles rupture-free survival')
    plt.tight_layout()
else:
    print('Run features/feature_store.py first')